# PhysX-Omni Paper Deep Reading - Step 6: Datasets

This notebook is the executable companion for Step 6. It explains why the datasets are needed, where they come from, how they are built, how large they are, and how they enter training and evaluation.

Suggested order: read the two embedded Markdown notes first, then run the evidence cells below.

## 1. Main Dataset Reading

The following section is loaded from `physx_omni_paper_datasets_step6.md`.

# PhysX-Omni 论文精读 第六步：数据集设计、来源、数量与构建方法

论文地址：[https://arxiv.org/abs/2605.21572v1](https://arxiv.org/abs/2605.21572v1)  
代码地址：[https://github.com/physx-omni/PhysX-Omni](https://github.com/physx-omni/PhysX-Omni)  
PhysXVerse：[https://huggingface.co/datasets/PhysX-Omni/PhysXVerse](https://huggingface.co/datasets/PhysX-Omni/PhysXVerse)  
PhysX-Mobility：[https://huggingface.co/datasets/Caoza/PhysX-Mobility](https://huggingface.co/datasets/Caoza/PhysX-Mobility)  
PhysXNet / PhysX-3D：[https://huggingface.co/datasets/Caoza/PhysX-3D](https://huggingface.co/datasets/Caoza/PhysX-3D)  
PhysX-Bench：[https://huggingface.co/datasets/PhysX-Omni/PhysX-Bench](https://huggingface.co/datasets/PhysX-Omni/PhysX-Bench)

## 0. 这一节先分清楚“数据集”的两层含义

PhysX-Omni 里有两类数据：

1. **训练数据**：用于训练 PhysX-Omni 的 simulation-ready physical 3D assets。核心来源是 PhysXNet、PhysX-Mobility、PhysXVerse，论文说合计超过 `42K` 个物理 3D 资产。
2. **评测数据**：用于无 GT benchmark 的 PhysX-Bench，核心是 `1214` 张 condition images 和 description JSON。第五步已经展开过 Bench，本节只把它放进完整数据版图里。

本文真正新增、最值得重点讲的是 **PhysXVerse**。它是作者为解决数据稀缺和类别不够广的问题构建的 general simulation-ready physical 3D dataset。

## 1. 为什么要新建 PhysXVerse

论文认为现有 3D 生成数据主要有三个问题：

- **几何/外观多，物理少**：很多 3D 数据集有 mesh、texture、render image，但没有绝对尺度、材料、affordance、关节、功能描述等可仿真属性。
- **单一资产类型多，统一物理资产少**：已有工作常常只处理 articulated object 或 deformable object 的某一个子问题，很少统一 rigid、deformable、articulated。
- **类别覆盖窄**：PhysX-Anything / PhysXGen 已经开始做 physical 3D generation，但训练数据多样性仍限制了泛化能力。

因此 PhysXVerse 的目标不是再做一个“看起来像”的 3D 数据集，而是做一个可以训练模型输出 simulation-ready asset 的数据集。每个对象不仅有部件和几何，还要有：

- 真实世界尺度。
- 每个 part 的材料、密度、弹性模量、泊松比。
- part-level affordance rank。
- part-level 基础描述。
- group-level 运动学关系和关节参数。

## 2. 数据集家族总览

| 数据集 | 角色 | 来源 | 论文/卡片中关键数量 | 主要用途 |
|---|---|---|---:|---|
| PhysXNet / PhysX-3D | 早期 physics-grounded 3D 数据 | 基于 PartNet / ShapeNet 体系 | PhysX-3D 论文提到 PhysXNet 约 `26K` assets；HF 当前含 PhysXNet / PhysXNet-XL 大量 zip | 提供物理属性标注和基础 JSON/URDF/XML 结构 |
| PhysX-Mobility | articulated/simulation-ready 数据 | 基于 PartNet-Mobility | PhysX-Anything 称 `2K+` common objects；补充材料给出 `1636` train、`388` test | 增强 articulated / mobility 类型数据 |
| PhysXVerse | 本文新增 general simulation-ready 数据 | 从 PartVerse 筛选、清洗、物理标注 | `8.7K+` assets，`2.9K+` categories，part count `1-65` | 扩大类别和结构复杂度，训练 PhysX-Omni |
| PhysX-Bench | 评测条件图数据 | real-world photos + synthetic rendered images | HF 当前 `1214` images：426 inthewild / 388 mobility / 400 verse | 无 GT 评测 |

训练时作者把 PhysXNet、PhysX-Mobility、PhysXVerse 合并，形成 `42K+` simulation-ready physical 3D assets。每个 object 渲染 `25` 张 conditioning images，因此 object-view 级训练输入规模至少是：

```text
42K objects * 25 views > 1.05M object-view conditioning images
```

实际 VLM 训练样本还会按 part 展开，所以监督条目数会进一步乘以部件数量。

## 3. PhysXVerse 的来源：PartVerse

PhysXVerse 从 PartVerse 筛选和再标注而来。PartVerse 本身来自 CoPart 工作，定位是大规模 3D part dataset：从 Objaverse 中经过 mesh segmentation、人工后处理，并给出 part-level caption。公开资料提到 PartVerse 约 `12K` objects、`91K` parts。

PhysX-Omni 不是直接拿 PartVerse 当训练数据，而是把它进一步变成 simulation-ready physical 3D data：

```mermaid
flowchart LR
  A["PartVerse: part-segmented 3D assets"] --> B["filter invalid samples"]
  B --> C["merge tiny/noisy parts"]
  C --> D["render multi-view images"]
  D --> E["GPT/VLM preliminary physical annotations"]
  E --> F["human verification and refinement"]
  F --> G["PhysXVerse: sim-ready physical 3D assets"]
```

这一层变化很关键：PartVerse 更偏 part-level geometry / caption，PhysXVerse 增加的是物理世界可用属性。

## 4. PhysXVerse 构建流程

论文和代码合起来可以还原出更具体流程：

### 4.1 清洗与结构整理

论文描述：

- 使用 PartVerse 的 human-verified segmentation annotations。
- 过滤 invalid samples。
- 合并 excessively small or noisy parts，提升结构一致性。

对应目的：

- 减少碎片 part。
- 避免 tiny part 导致 voxel token 过长但语义价值低。
- 保证 group_info 和 part-level geometry 更稳定。

### 4.2 多视角渲染

论文训练设置：

- 每个 object 渲染 `25` 张 conditioning images。
- 这些图片用于训练 VLM 从单图/多视角视觉条件理解几何、part 和物理属性。

README 也写明：

- PhysX-Mobility 和 PhysXVerse 使用 PhysX-Anything 的 `render_cond_mobility.py` 风格工具生成 conditioning images。
- PhysXNet 使用 PhysX-3D 侧的 preprocessing/rendering 工具。

### 4.3 物理属性初标注

论文描述：

- 渲染多视角 images。
- 用强 VLM，也就是 GPT，生成 preliminary physical annotations。
- 标注内容包括 absolute scale、affordance、material、functional descriptions、kinematic information。

这一步不是最终答案，而是自动初稿。

### 4.4 人工验证与 refinement

论文强调 human-in-the-loop：

- 自动标注后由 human annotators verify and refine。
- 目标是保证 physical plausibility 和 annotation quality。

这说明 PhysXVerse 的价值不只是大规模，而是“VLM 初标注 + 人工审校”的可扩展质量控制。

### 4.5 几何归一化与体素化

本地脚本：

```text
dataset/1voxel_verse.py
```

核心处理：

- 读取 `PhysXVerse/partobj/<object_id>/<part_id>/<part_id>.obj`。
- 合并所有 part mesh，计算整体 bbox。
- 把 object 缩放和平移到统一坐标系，大致归一到 `[-0.5, 0.5]`。
- 做一个 `pi/2` 的 x-axis rotation。
- 同步变换 `group_info` 中的关节方向、轴位置、hinge position 等参数。
- 对每个 part 分别体素化，分辨率包括 `16 / 32 / 64`。
- 保存 `ind_<part>.npy` 和 `allind.npy`。

这一步把原始 mesh 数据变成 PhysX-Omni 需要的显式 3D voxel 监督。

### 4.6 JSON 到结构化文本

本地脚本：

```text
dataset/2encode_representation_64_finetune.py
```

它把 `finaljson/<object_id>.json` 转成模型可读的文本描述：

```text
Name: ...
Category: ...
Dimension: ...
Parts:
l_0: <part name>| <affordance rank>| <material>| <density>| <Young>| <Poisson>| <description>
Group_info:
group_1: [l_0, ...]; Type: C relative to group_0; Param: direction..., axis position..., revolute range...
```

这个文本正是 VLM 第一阶段要学会输出的“structured physical description”。

### 4.7 Template-RLE 几何监督

本地脚本：

```text
dataset/3generate_data_new_64_finetune_rle.py
```

它把每个 part 的 `64^3` voxel 编码成 z-slice 2D RLE，并进一步做 template-based lossless compression：

- 每个 z slice 先变成 2D run-length list。
- 相似 slice 共享 template。
- 每一层记录 template label，加上 adds / removes delta。
- 代码里有 decode 和 roundtrip equality 检查，确认编码是 lossless。

最后每个训练样本形成 Qwen-VL 对话：

```json
{
  "id": "<object_id>_<view_id>",
  "image": "<object_id>/<view_id>.png",
  "data_source": "physx",
  "conversations": [
    {"from": "human", "value": "<image>\\n... structured description prompt ..."},
    {"from": "gpt", "value": "... structured description ..."},
    {"from": "human", "value": "Based on the structured description of l_i, generate its 3D voxel ..."},
    {"from": "gpt", "value": "... template-RLE voxel ..."}
  ],
  "meshlength": "<token count>"
}
```

代码会过滤 `tokennum >= 20000` 的样本，以适配长上下文训练预算。

## 5. PhysXVerse 数据 schema

一个对象 JSON 的顶层结构：

| 字段 | 含义 |
|---|---|
| `object_name` | 对象名 |
| `category` | 类别 |
| `dimension` | 真实物理尺寸，通常是 `L*W*H` cm |
| `parts` | part-level 物理和语义属性 |
| `group_info` | group-level 运动学关系 |

`parts[]` 典型字段：

| 字段 | 含义 |
|---|---|
| `label` | part id |
| `name` | part 名称 |
| `material` | 材料 |
| `density` | 密度 |
| `priority_rank` | affordance rank，通常 1-10 |
| `Basic_description` | part 基础描述 |
| `Young's Modulus (GPa)` | 杨氏模量 |
| `Poisson's Ratio` | 泊松比 |

`group_info` 的运动类型：

| Type | 含义 | 例子 |
|---|---|---|
| A | unconstrained / freely movable | 瓶中水等无固定约束对象 |
| B | prismatic / slide | 抽屉 |
| C | revolute / axis rotation | 门 |
| D | hinge around point | 软管等 hinge-like motion |
| CB | prismatic + revolute | 既能旋转又能滑动的盖子 |
| E | fixed / rigid to world or base group | 固定基座 |

更详细 schema 和脚本字段见：

`C:\Users\robot\Documents\成长学习库\physx_omni_step6_dataset_schema_pipeline.md`

## 6. 当前公开资产体积

我用 Hugging Face API 核对了当前公开仓库结构：

| HF repo | 文件结构 | 当前体积 |
|---|---|---:|
| `PhysX-Omni/PhysXVerse` | `PhysXVerse.zip.part_aa/ab/ac` + `merge.sh` | 约 `104.9 GB` |
| `Caoza/PhysX-Mobility` | `PhysX-Mobility.zip` | 约 `0.87 GB` |
| `Caoza/PhysX-3D` | PhysXNet / PhysXNet-XL 多个 zip 和分卷 zip | 约 `1.83 TB` |
| `PhysX-Omni/PhysX-Bench` | `demo_*/*.png` + `descript_*.json` | 约 `0.87 GB` |

注意：这些是当前 HF 文件体积，不等于论文训练样本数量。论文数量以 asset/object 计，文件体积还受 texture、mesh、render、分卷压缩影响。

## 7. 训练数据怎么接入模型

训练配置在：

```text
qwen-vl-finetune/qwenvl/data/__init__.py
qwen-vl-finetune/scripts/train_physx.sh
```

代码里三路数据名：

```text
physxverse64
physxnet64
physxmobility64
```

每路数据都有：

```python
{
  "annotation_path": "...",  # generated training json
  "data_path": "...",        # rendered conditioning image path
}
```

训练脚本中：

```text
datasets=physxverse64,physxnet64,physxmobility64
model=Qwen/Qwen2.5-VL-7B-Instruct
num_train_epochs=5
learning_rate=2e-5
model_max_length=16384
```

论文说实际训练使用 `64 NVIDIA A100 GPUs` 约 `14` 天，有效 batch size `128`。本地脚本是集群模板，`SBATCH --gres=gpu:8` 只是单节点示例，不代表论文完整训练规模。

## 8. 这套数据设计的核心价值

PhysX-Omni 的数据设计有三个关键点：

1. **把物理属性变成可学习的语言结构**  
   VLM 学到的不只是“这是什么物体”，而是对象名、类别、尺度、part、材料、affordance、关节组和运动参数。

2. **把几何监督变成 VLM 可输出的文本序列**  
   通过 template-RLE，`64^3` voxel 被压进普通文本 token，不需要额外 special tokens 或专门 tokenizer。

3. **把 part-level 结构和 simulation-ready 属性绑定**  
   每个 part 有物理属性，每个 group 有运动关系。这比单个整体 mesh 更适合仿真、机器人交互和下游 scene construction。

## 9. 严谨边界

当前本地没有完整解压 PhysXVerse / PhysXNet / PhysX-Mobility。第六步基于：

- 论文源码。
- 官方 GitHub README 和预处理脚本。
- Hugging Face 当前仓库元数据。
- 本地 demo 输出的 `basic_info.json` 作为 schema 实例。

所以本节可以讲清楚数据设计和构建流程，但不能声称已经完成三大训练数据集的完整本地校验或统计复算。



## 2. Dataset Schema and Code Pipeline

The following section is loaded from `physx_omni_step6_dataset_schema_pipeline.md`.

# PhysX-Omni 第六步附录：数据 schema 与预处理管线

## 1. 数据目录结构

PhysXVerse 在本地预处理脚本中假设解压后结构大致为：

```text
PhysXVerse/
  partobj/
    <object_id>/
      <part_id>/
        <part_id>.obj
  finaljson/
    <object_id>.json
```

预处理临时输出：

```text
tmp_verse/
  finaljson/<object_id>.json
  partobj/<object_id>/<res>/
    mesh_new_<part_id>.ply
    ind_<part_id>.npy
    allind.npy
    alldict_vert.pkl
```

结构化文本输出：

```text
txt_rep_64_finetune_verse/<object_id>.txt
```

训练 JSON 输出：

```text
trainset_64_final_verse/training_set_<shard_id>_randompart.json
```

## 2. JSON schema

顶层：

```json
{
  "object_name": "Dumpster",
  "category": "Waste Container",
  "dimension": "180*120*150",
  "parts": [],
  "group_info": {}
}
```

part：

```json
{
  "label": 0,
  "name": "Caster Wheel",
  "material": "Steel with Rubber",
  "density": 4.45,
  "priority_rank": 7,
  "Basic_description": "A caster wheel attached to the bottom.",
  "Young's Modulus (GPa)": 100.005,
  "Poisson's Ratio": 0.395
}
```

group：

```json
{
  "0": [1, 4],
  "1": [
    [0],
    "0",
    [0.0, 0.0, 1.0, 0.203, 0.401, -0.5, -1.0, 1.0],
    "C"
  ]
}
```

解释：

- group `0` 通常是 base group。
- 非 0 group 形如 `[child_parts, parent_group, params, type]`。
- `params` 的含义依赖 `type`。

## 3. 运动类型参数

| Type | 名称 | 参数解释 |
|---|---|---|
| A | free / no movement constraints | 无显式参数 |
| B | prismatic | direction + slide range |
| C | revolute | axis direction + axis position + revolute range |
| D | hinge point | hinge position |
| CB | revolute + prismatic | rotate axis/position/range + slide direction/range |
| E | fixed | base/fixed group |

脚本里会把归一化变换同步应用到这些参数，避免 mesh 坐标变了但关节轴还在旧坐标。

## 4. `1voxel_verse.py`

主要任务：

1. 读取 `partobj/<object_id>` 下的 part OBJ。
2. 合并成整体 mesh，计算 bbox。
3. 归一化到统一尺度和中心。
4. 对关节参数做相同坐标变换。
5. 逐 part 生成 voxel indices。
6. 输出多个分辨率：`16 / 32 / 64`。

关键函数：

| 函数 | 作用 |
|---|---|
| `movtran()` | scale + translate + rotate mesh |
| `transfer()` | 对关节点/方向应用相同矩阵 |
| `generate_voxel()` | 用 Open3D point cloud 采样和 voxel grid 转 voxel indices |
| `load_obj_geometry_fast()` | 快速读取 OBJ 顶点和面 |

输出字段：

| 文件 | 含义 |
|---|---|
| `ind_<part>.npy` | 当前 part 的 voxel `(x,y,z)` indices |
| `allind.npy` | 全对象 voxel indices 拼接 |
| `mesh_new_<part>.ply` | 归一化后的 part mesh |
| `alldict_vert.pkl` | part 到 voxel vertex 的映射 |

## 5. `2encode_representation_64_finetune.py`

主要任务：

- 把 object JSON 转成训练 prompt 使用的结构化文本。
- 把物理参数和运动参数显式写出来。
- 对 B/C/CB 的方向向量做归一化。
- 把 slide range 转换到 voxel grid 单位。
- 把 revolute range 转成 degree。

文本模板：

```text
Name: <object name>
Category: <object category>
Dimension: <physical dimensions>
Parts:
l_0: <part name>| <affordance rank>| <material>| <density>| <young>| <poisson>| <description>
Group_info:
group_1: [l_0]; Type: C relative to group_0; Param: direction: [...], axis position: [...], revolute range: [...]
```

这里的文本和 `dataset/example_64_finetune_rle.txt` 保持同一 schema。

## 6. `3generate_data_new_64_finetune_rle.py`

主要任务：

1. 读取结构化文本。
2. 读取每个 part 的 `ind_<part>.npy`。
3. 把 voxel `(x,y,z)` 按 z slice 编成 2D RLE。
4. 对相似 z slice 做 template sharing。
5. 记录 adds/removes delta，保持 lossless。
6. 构造 Qwen-VL conversation 格式。
7. 用 tokenizer 统计 token 长度，过滤过长样本。

训练样本结构：

```json
{
  "id": "<object_id>_<view_id>",
  "image": "<object_id>/<view_id>.png",
  "conversations": [
    {"from": "human", "value": "<image>\\n..."},
    {"from": "gpt", "value": "... structured text ..."},
    {"from": "human", "value": "Based on the structured description of l_i, generate its 3D voxel ..."},
    {"from": "gpt", "value": "... RLE geometry ..."}
  ],
  "data_source": "physx",
  "meshlength": 12345
}
```

代码中每个 object、每个 part、每个 view 都会生成训练条目：

```text
for part in range(num_parts):
  for ind in range(0, 25):
    create one training sample
```

这解释了为什么 object 数是 42K+，但训练 JSON 条目会远高于 42K。

## 7. Template-RLE 具体机制

`encode_voxel_2drle_by_z()`：

- 输入 `(N,3)` voxel coords。
- 以 z 为 slice。
- 每个 z slice 把 `(x,y)` 展平成 `idx2d = x + W*y`。
- 连续 idx 变成 `(start, length)` run。

`runs_by_z_to_string_lossless()`：

- 为相似 slice 建 template，label 为 `a,b,c,...`。
- 每层引用某个 template。
- 如果有差异，用 `+[adds]` 和 `-[removes]` 表达。

`string_to_runs_by_z_lossless()` 和 `decode_voxel_2drle_by_z()`：

- 把文本还原成 voxel coords。
- 脚本里对 `coords` 和 `coords_rec` 做 equality check。

## 8. 训练配置字段

`qwen-vl-finetune/qwenvl/data/__init__.py`：

```python
PHYSXVERSE64_FINAL = {
    "annotation_path": "./",
    "data_path": "./PhysXVerse/renders",
}
PHYSXNET64_FINAL = {
    "annotation_path": "./",
    "data_path": "./PhysXNet/renders",
}
PHYSXMOBILITY64_FINAL = {
    "annotation_path": "./",
    "data_path": "./PhysX-mobility/renders",
}
```

`qwen-vl-finetune/scripts/train_physx.sh`：

```bash
datasets=physxverse64,physxnet64,physxmobility64
llm=Qwen/Qwen2.5-VL-7B-Instruct
lr=2e-5
model_max_length=16384
```

论文训练细节：

| 项 | 值 |
|---|---|
| VLM backbone | Qwen2.5-VL-7B-Instruct |
| Epochs | 5 |
| GPUs | 64 NVIDIA A100 |
| Time | about 14 days |
| Peak LR | `2e-5` |
| Schedule | cosine decay |
| Warmup ratio | `0.03` |
| Effective batch size | 128 |
| Max sequence length | 16384 |

## 9. Split 与本地证据

本地代码包含：

```text
dataset/testset_physxverse.npy
```

该文件当前有 `400` 个 object id。`2encode_representation_64_finetune.py` 会跳过这些 test ids，不把它们写入训练文本：

```python
testset = np.load('testset_physsxverse.npy')
if name in testset:
    skip
```

注意脚本里的文件名拼写是 `testset_physsxverse.npy`，而本地实际文件是 `testset_physxverse.npy`。这可能是公开代码中的小拼写问题，真正跑预处理时需要确认或修正路径。

## 10. Dataset 与模型能力的对应关系

| 数据字段 | 模型学习到的能力 |
|---|---|
| `dimension` | 估计真实物体尺度 |
| `parts` | 识别对象组成部件 |
| `material/density/Young/Poisson` | 生成材料和力学属性 |
| `priority_rank` | 生成 affordance heatmap / interaction prior |
| `Basic_description` | part-level semantic grounding |
| `group_info` | articulated structure 和 joint constraints |
| `ind_<part>.npy` + RLE | part-level explicit geometry |
| 25 views | 提高视角鲁棒性和单图泛化 |



## 3. Executable Check: Local Paths and Key Assets

This cell verifies that the Step 6 notes, official code, test split, and reproduction output are readable on this machine.

In [1]:

from pathlib import Path
import json
import textwrap

ROOT = Path.cwd()
CODE = ROOT / 'physx-omni-assets' / 'code' / 'PhysX-Omni'
DATASET = CODE / 'dataset'
QWEN_DATA = CODE / 'qwen-vl-finetune' / 'qwenvl' / 'data' / '__init__.py'
TRAIN_SH = CODE / 'qwen-vl-finetune' / 'scripts' / 'train_physx.sh'
OFFICIAL_OUT = Path(r'C:\Users\robot\physx_outputs\official_demo_full')

paths = {
    'Step 6 main markdown': ROOT / 'physx_omni_paper_datasets_step6.md',
    'Step 6 schema markdown': ROOT / 'physx_omni_step6_dataset_schema_pipeline.md',
    'Official code repo': CODE,
    'Dataset scripts dir': DATASET,
    'PhysXVerse test split npy': DATASET / 'testset_physxverse.npy',
    'Example RLE prompt schema': DATASET / 'example_64_finetune_rle.txt',
    'Qwen data registry': QWEN_DATA,
    'Training shell script': TRAIN_SH,
    'Official demo output schema': OFFICIAL_OUT / 'basic_info.json',
}

for name, path in paths.items():
    print(f'{name:32s} | exists={path.exists()} | {path}')


Step 6 main markdown             | exists=True | C:\Users\robot\Documents\成长学习库\physx_omni_paper_datasets_step6.md
Step 6 schema markdown           | exists=True | C:\Users\robot\Documents\成长学习库\physx_omni_step6_dataset_schema_pipeline.md
Official code repo               | exists=True | C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni
Dataset scripts dir              | exists=True | C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\dataset
PhysXVerse test split npy        | exists=True | C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\dataset\testset_physxverse.npy
Example RLE prompt schema        | exists=True | C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\dataset\example_64_finetune_rle.txt
Qwen data registry               | exists=True | C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\qwen-vl-finetune\qwenvl\data\__init__.py
Training shell script            | exists=True | C:\Users\robot\Documents\成

## 4. Executable Check: Public Hugging Face Assets

This cell reads public Hugging Face Hub metadata: file count, total size, modified time, and largest files. It also probes the Dataset Viewer API; many of these repos are zip-based assets rather than tabular viewer datasets.

In [2]:

from huggingface_hub import HfApi
import requests

repos = [
    ('PhysXVerse', 'PhysX-Omni/PhysXVerse'),
    ('PhysX-Mobility', 'Caoza/PhysX-Mobility'),
    ('PhysX-3D / PhysXNet', 'Caoza/PhysX-3D'),
    ('PhysX-Bench', 'PhysX-Omni/PhysX-Bench'),
]
api = HfApi()

def fmt_gb(num_bytes):
    if num_bytes is None:
        return 'n/a'
    return f'{num_bytes / (1024 ** 3):.3f} GB'

rows = []
for label, repo_id in repos:
    try:
        info = api.repo_info(repo_id=repo_id, repo_type='dataset', files_metadata=True)
        siblings = list(info.siblings or [])
        total_bytes = sum((getattr(s, 'size', None) or 0) for s in siblings)
        biggest = sorted(
            [(getattr(s, 'rfilename', ''), getattr(s, 'size', None) or 0) for s in siblings],
            key=lambda item: item[1],
            reverse=True,
        )[:5]
        rows.append({
            'label': label,
            'repo_id': repo_id,
            'sha': getattr(info, 'sha', '')[:12],
            'last_modified': str(getattr(info, 'last_modified', None) or getattr(info, 'lastModified', None)),
            'file_count': len(siblings),
            'total_size': fmt_gb(total_bytes),
            'biggest': biggest,
            'error': None,
        })
    except Exception as exc:
        rows.append({
            'label': label,
            'repo_id': repo_id,
            'sha': '',
            'last_modified': '',
            'file_count': None,
            'total_size': 'n/a',
            'biggest': [],
            'error': repr(exc),
        })

print('| dataset | repo | sha | files | total size | last modified |')
print('|---|---|---:|---:|---:|---|')
for row in rows:
    print(f"| {row['label']} | {row['repo_id']} | {row['sha']} | {row['file_count']} | {row['total_size']} | {row['last_modified']} |")
    if row['error']:
        print('  error:', row['error'])

print('\nLargest files:')
for row in rows:
    print(f"\n{row['label']} ({row['repo_id']}):")
    for filename, size in row['biggest']:
        print(f'  {filename:60s} {fmt_gb(size)}')

print('\nDataset Viewer API /is-valid probe:')
for label, repo_id in repos:
    try:
        resp = requests.get(
            'https://datasets-server.huggingface.co/is-valid',
            params={'dataset': repo_id},
            timeout=20,
        )
        print(f'{repo_id:28s} status={resp.status_code} body={resp.text[:220]}')
    except Exception as exc:
        print(f'{repo_id:28s} error={exc!r}')


| dataset | repo | sha | files | total size | last modified |
|---|---|---:|---:|---:|---|
| PhysXVerse | PhysX-Omni/PhysXVerse | 264d5864d0e6 | 6 | 104.867 GB | 2026-05-20T18:52:47.000Z |
| PhysX-Mobility | Caoza/PhysX-Mobility | d0768ee9e141 | 3 | 0.873 GB | 2025-12-11T03:31:43.000Z |
| PhysX-3D / PhysXNet | Caoza/PhysX-3D | ca1f6d3f5cfb | 97 | 1828.754 GB | 2025-11-11T07:35:19.000Z |
| PhysX-Bench | PhysX-Omni/PhysX-Bench | 6b4eb29a7016 | 1219 | 0.870 GB | 2026-05-20T06:10:32.000Z |

Largest files:

PhysXVerse (PhysX-Omni/PhysXVerse):
  PhysXVerse.zip.part_aa                                       40.000 GB
  PhysXVerse.zip.part_ab                                       40.000 GB
  PhysXVerse.zip.part_ac                                       24.867 GB
  .gitattributes                                               0.000 GB
  README.md                                                    0.000 GB

PhysX-Mobility (Caoza/PhysX-Mobility):
  PhysX-Mobility.zip                                 

PhysX-Omni/PhysXVerse        status=200 body={"preview":false,"viewer":false,"search":false,"filter":false,"statistics":false}


Caoza/PhysX-Mobility         status=200 body={"preview":false,"viewer":false,"search":false,"filter":false,"statistics":false}


Caoza/PhysX-3D               status=200 body={"preview":false,"viewer":false,"search":false,"filter":false,"statistics":false}


PhysX-Omni/PhysX-Bench       status=200 body={"preview":true,"viewer":false,"search":false,"filter":false,"statistics":false}


## 5. Executable Check: PhysXVerse Test Split

The official code includes `testset_physxverse.npy`. This is not the full PhysXVerse dataset; it is a 400-object id split that aligns with the PhysX-Bench `demo_verse` subset.

In [3]:

import numpy as np

testset_path = DATASET / 'testset_physxverse.npy'
arr = np.load(testset_path)
print('path:', testset_path)
print('shape:', arr.shape)
print('dtype:', arr.dtype)
print('first 12 ids:')
for item in arr[:12].tolist():
    print(' ', item)


path: C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\dataset\testset_physxverse.npy
shape: (400,)
dtype: <U32
first 12 ids:
  00d1cb5aa82745228a3b764c97f867de
  0134b71129524e32a199ca0963c7441a
  02dcf7624c21467fb391e4303823f8e6
  0305b7bf2650453d8aca2639b9a3c5ca
  0323b1df8d364c5e9227489b8c0cbcda
  03ba684b8bad4fffa1a8cc2da2c2c59f
  03e65b60334d4d01bdceb292d362b743
  0416f30815d6422791746b379f802405
  04a5b9260a71412cb860357610c80e35
  056ccf898f9e49acb6e3e370deabf184
  05e6f8ed25e74242957b34f9256ffc47
  068f17e7bcf74d04bacf5524c4a2079d


## 6. Executable Check: Training Text/RLE Schema

`example_64_finetune_rle.txt` is the most direct format description. It combines object/category/dimension, part-level attributes, assembly relations, and per-part 64^3 voxel RLE supervision.

In [4]:

example_path = DATASET / 'example_64_finetune_rle.txt'
print(example_path.read_text(encoding='utf-8'))


Please analyze the given image of an object and output its structured description in the following format (voxel grid=64):
Name: <object name>
Category: <object category>
Dimension: <physical dimensions in cm like 50*40*30>
Parts:
l_<id>: <part name>| <Affordance rank>| <Material Name>| <Density>| <Young's Modulus>| <Poisson's Ratio>| <basic description of the part>
l_<id>: <part name>| <Affordance rank>| <Material Name>| <Density>| <Young's Modulus>| <Poisson's Ratio>| <basic description of the part>
...
Group_info:
group_<id>: [l_<id>, ...] (child); Type: E; Params: N/A
group_<id>: [l_<id>, ...] (child); Type: A relative to group_<id> (parent); Params: N/A
group_<id>: [l_<id>, ...] (child); Type: B relative to group_<id> (parent); Params: direction: [x,y,z], slide range (in voxel grid): [min,max]
group_<id>: [l_<id>, ...] (child); Type: C relative to group_<id> (parent); Params: direction: [x,y,z], axis position (in voxel grid): [x,y,z], revolute range (degree): [min,max]
group_<id>:

## 7. Executable Check: JSON Fields from Official Reproduction Output

This cell reads the previously generated `basic_info.json` from the official demo reproduction and checks whether the model output schema aligns with the training data schema.

In [5]:
basic_info_path = OFFICIAL_OUT / 'basic_info.json'
basic = json.loads(basic_info_path.read_text(encoding='utf-8'))
print('top-level keys:', list(basic.keys()))
print('object/category/dimension:', basic.get('object_name'), basic.get('category'), basic.get('dimension'))

parts = basic.get('parts', [])
print('\nparts count:', len(parts))
for part in parts[:4]:
    compact = {k: part.get(k) for k in ['label', 'name', 'material', 'density', "Young's Modulus", "Poisson's Ratio", 'affordance', 'priority_rank'] if isinstance(part, dict) and k in part}
    print(compact)

groups_raw = basic.get('group_info', [])
if isinstance(groups_raw, dict):
    group_items = list(groups_raw.items())
elif isinstance(groups_raw, list):
    group_items = list(enumerate(groups_raw))
else:
    group_items = [('raw', groups_raw)]
print('\ngroup_info container:', type(groups_raw).__name__)
print('group_info count:', len(group_items))
for key, group in group_items[:4]:
    print(key, group)


top-level keys: ['object_name', 'category', 'dimension', 'parts', 'group_info']
object/category/dimension: Dumpster Waste Container 180*120*150

parts count: 7
{'label': 0, 'name': 'Caster Wheel (Front Right)', 'material': 'Steel with Rubber', 'density': 4.45, "Poisson's Ratio": 0.395, 'priority_rank': 7}
{'label': 1, 'name': 'Main Body', 'material': 'Steel', 'density': 7.8, "Poisson's Ratio": 0.3, 'priority_rank': 10}
{'label': 2, 'name': 'Lid', 'material': 'Plastic (HDPE)', 'density': 0.95, "Poisson's Ratio": 0.4, 'priority_rank': 1}
{'label': 3, 'name': 'Caster Wheel (Front Left)', 'material': 'Steel with Rubber', 'density': 4.45, "Poisson's Ratio": 0.395, 'priority_rank': 8}

group_info container: dict
group_info count: 6
0 [1, 4]
1 [[0], '0', [0.0, 0.0, 1.0, 0.20328124999999997, 0.400625, -0.5, -1.0, 1.0], 'C']
2 [[2], '0', [0.0, 1.0, 0.0, -0.2915625, -0.5, 0.3203125, -1.0, 0.0], 'C']
3 [[3], '0', [0.0, 0.0, 1.0, 0.20265624999999998, -0.38828125, -0.5, -1.0, 1.0], 'C']


## 8. Executable Check: Preprocessing Scripts and Training Config

This cell extracts key lines from the official code: data registration, training shell parameters, and the three PhysXVerse preprocessing scripts.

In [6]:

def print_matching_lines(path, terms, context_name):
    print(f'\n## {context_name}: {path}')
    text = path.read_text(encoding='utf-8', errors='replace').splitlines()
    for idx, line in enumerate(text, start=1):
        if any(term in line for term in terms):
            print(f'{idx:04d}: {line}')

print_matching_lines(
    QWEN_DATA,
    ['PHYSXVERSE64_FINAL', 'PHYSXNET64_FINAL', 'PHYSXMOBILITY64_FINAL', 'physxverse64', 'physxnet64', 'physxmobility64', 'data_path'],
    'Qwen-VL data registry',
)

print_matching_lines(
    TRAIN_SH,
    ['datasets=', 'Qwen/Qwen2.5-VL-7B-Instruct', 'learning_rate', 'lr=', 'model_max_length', 'num_train_epochs', 'warmup_ratio'],
    'Training script',
)

for script_name, terms in [
    ('1voxel_verse.py', ['PhysXVerse', 'partobj', 'finaljson', 'voxel', 'scale', 'offset', 'group_info']),
    ('2encode_representation_64_finetune.py', ['testset', 'Name:', 'Category:', 'Dimension:', 'Group_info', 'Movement_description']),
    ('3generate_data_new_64_finetune_rle.py', ['rle', 'meshlength', 'image', 'conversations', 'tokennum', '20000', 'grid=64']),
]:
    print_matching_lines(DATASET / script_name, terms, script_name)



## Qwen-VL data registry: C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\qwen-vl-finetune\qwenvl\data\__init__.py
0006:     "data_path": "",
0011:     "data_path": f"",
0016:     "data_path": "PATH_TO_MP_DOC_DATA",
0021:     "data_path": "PATH_TO_CLEVR_MC_DATA",
0026:     "data_path": "PATH_TO_VIDEOCHATGPT_DATA",
0033: PHYSXVERSE64_FINAL = {
0035:     "data_path": "./PhysXVerse/renders",  # rendered image path
0037: PHYSXNET64_FINAL = {
0039:     "data_path": "./PhysXNet/renders",  
0042: PHYSXMOBILITY64_FINAL = {
0044:     "data_path": "./PhysX-mobility/renders",  
0054:     "physxverse64":PHYSXVERSE64_FINAL,
0055:     "physxnet64":PHYSXNET64_FINAL,
0056:     "physxmobility64":PHYSXMOBILITY64_FINAL,

## Training script: C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\qwen-vl-finetune\scripts\train_physx.sh
0027: llm=Qwen/Qwen2.5-VL-7B-Instruct
0030: lr=2e-5
0038: datasets=physxverse64,physxnet64,physxmobility64
0055:     --num_train_epochs 5 \
0065:

## 9. Scale Estimate: Why 25-view Inputs and RLE Compression

The paper and code point to two constraints: multi-view images constrain visible and hidden geometry, while dense 64^3 voxel output would be too long for direct text generation. Structured text plus per-part RLE is the chosen bridge.

In [7]:

physxverse_assets_lower = 8700
training_assets_lower = 42000
views_per_asset = 25
voxel_grid = 64 ** 3

print(f'PhysXVerse assets lower bound: >{physxverse_assets_lower:,}')
print(f'PhysXVerse 25-view conditioning images lower bound: >{physxverse_assets_lower * views_per_asset:,}')
print(f'Combined training assets lower bound: >{training_assets_lower:,}')
print(f'Combined 25-view conditioning images lower bound: >{training_assets_lower * views_per_asset:,}')
print(f'Dense 64^3 voxel cells per part before compression: {voxel_grid:,}')
print('RLE role: encode sparse voxel runs as start_index + length, reducing sequence pressure while preserving a text supervision target for the LLM.')


PhysXVerse assets lower bound: >8,700
PhysXVerse 25-view conditioning images lower bound: >217,500
Combined training assets lower bound: >42,000
Combined 25-view conditioning images lower bound: >1,050,000
Dense 64^3 voxel cells per part before compression: 262,144
RLE role: encode sparse voxel runs as start_index + length, reducing sequence pressure while preserving a text supervision target for the LLM.


## 10. Step 6 Conclusion

PhysX-Omni's dataset design can be summarized as follows: PhysXVerse broadens general object coverage; PhysX-Mobility and PhysXNet supply movable and physical-property supervision; PhysX-Bench organizes evaluation images into wild/mobility/verse subsets; and 25-view images plus structured physical text plus per-part voxel RLE unify the data into a trainable input/output format.

This notebook connects the paper claims, public data assets, local code, and local reproduction output in one evidence chain.